# Dispersion of cylindrical TM modes in a magnetized plasma column

This notebook studies the dispersion properties of transverse magnetic (TM) modes in a cylindrical plasma column, under conditions where the plasma is confined by geometry and the axial propagation is described by a longitudinal wavenumber $k$.

The analysis is organized around three complementary descriptions of the same physical system:

1. a **closed-form cold-plasma dispersion relation**;
2. a **warm-fluid extension** that incorporates thermal corrections through the Debye length;
3. a **numerical root-finding formulation** based on the corresponding residual equation.

The objective is to characterize the modal spectrum, identify the low- and high-frequency branches, and compare analytical and numerical solutions directly by **superposing them in the same figures**.

---

## Physical setting

We consider a homogeneous plasma column of radius $a$, with cylindrical symmetry and axial propagation. The radial confinement quantizes the transverse structure of the fields, so each mode is labeled by a pair of integers $(m,n)$ associated with the Bessel eigenvalue

$$
J_m(X_{mn}) = 0,
$$

where $X_{mn}$ is the $n$-th zero of the Bessel function of the first kind and order $m$.

The characteristic plasma frequency is the electron plasma frequency $\omega_{pe}$, and the natural thermal scale is the electron Debye length $\lambda_D$.

---

## Scope

The notebook develops the following program:

- formulation of the normalized dispersion relations;
- computation of physical parameters using **PlasmaPy**;
- analytical evaluation of modal branches;
- numerical recovery of the same branches from the residual equation;
- direct comparison between cold and warm models;
- quantitative error estimates between analytical and numerical solutions.

The final result is a reusable computational framework for examining cylindrical plasma-wave dispersion in both the cold and weakly thermal regimes.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import wofz
from scipy.optimize import root
from scipy.optimize import brentq
from scipy.special import jn_zeros
from scipy.optimize import minimize

import astropy.units as u
from astropy.constants import c
from IPython.display import display

from plasmapy.dispersion import plasma_dispersion_func, plasma_dispersion_func_deriv
from plasmapy.formulary.frequencies import plasma_frequency
from plasmapy.formulary.lengths import Debye_length

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
})

## 1. Normalization and dispersion relations

A dimensionless formulation is convenient because it isolates the structure of the dispersion relation from the specific dimensional scales. We define

$$
\bar\omega = \frac{\omega}{\omega_{pe}},
\qquad
\bar k = ka,
\qquad
\bar c = \frac{c}{a\omega_{pe}},
\qquad
\bar\lambda_D = \frac{\lambda_D}{a}.
$$

Here:

- $\omega$ is the angular frequency,
- $k$ is the axial wavenumber,
- $c$ is the speed of light,
- $a$ is the cylinder radius,
- $\omega_{pe}$ is the electron plasma frequency,
- $\lambda_D$ is the electron Debye length.

The transverse confinement enters through the eigenvalue $X_{mn}$.

---

### Cold-plasma model

In the cold limit, the normalized TM dispersion relation can be written as

$$
\bar\omega_{\pm}(\bar k)
=
\sqrt{
\frac{1}{2}
\left[
1 + \bar c^2(\bar k^2 + X_{mn}^2)
\pm
\sqrt{
\left(1 + \bar c^2(\bar k^2 + X_{mn}^2)\right)^2
- 4 \bar c^2 \bar k^2
}
\right]
}.
$$

This expression yields two branches:

- a **low-frequency branch** $\bar\omega_-$,
- a **high-frequency branch** $\bar\omega_+$.

The corresponding residual form is

$$
\bar k^2 - \frac{\bar\omega^2}{\bar c^2}
+ \frac{X_{mn}^2}{\epsilon(\bar\omega)} = 0,
\qquad
\epsilon(\bar\omega)=1-\frac{1}{\bar\omega^2}.
$$

This residual equation will be used for numerical root-finding.

---

### Warm-fluid model

To include finite-temperature effects at the fluid level, we introduce the warm dielectric response

$$
\epsilon(\bar\omega,\bar k)
=
1-\frac{1}{\bar\omega^2 - 3\bar\lambda_D^2 \bar k^2}.
$$

The dispersion equation becomes

$$
\bar k^2 - \frac{\bar\omega^2}{\bar c^2}
+ \frac{X_{mn}^2}{\epsilon(\bar\omega,\bar k)} = 0.
$$

This form can again be reduced to a quadratic equation in $y=\bar\omega^2$, leading to a closed-form warm-fluid solution. We will compare that exact warm-fluid expression against numerical roots of the same residual equation.

For reference, we also include an alternative approximate thermal expression often used in exploratory implementations, in order to quantify how closely it follows the exact warm-fluid result.


In [ ]:
# Reference plasma parameters
# These values are representative of laboratory plasma-column conditions.

n_e = (2.2e8 * u.cm**-3).to(u.m**-3)
T_e = (10 * u.eV).to(u.K, equivalencies=u.temperature_energy())
a = (5.2 * u.cm).to(u.m)

omega_pe = plasma_frequency(n_e, particle="e-")
f_pe = plasma_frequency(n_e, particle="e-", to_hz=True)
lambda_D = Debye_length(T_e, n_e)

cbar = c.to_value(u.m / u.s) / (a.to_value(u.m) * omega_pe.to_value(u.rad / u.s))
lambdaD_bar = (lambda_D / a).to_value(u.dimensionless_unscaled)

params_df = pd.DataFrame({
    "Parameter": [
        r"$n_e$",
        r"$T_e$",
        r"$a$",
        r"$\omega_{pe}$",
        r"$f_{pe}$",
        r"$\lambda_D$",
        r"$\bar c = c/(a\omega_{pe})$",
        r"$\bar\lambda_D = \lambda_D/a$",
    ],
    "Value": [
        f"{n_e:.3e}",
        f"{T_e.to(u.eV, equivalencies=u.temperature_energy()):.3f}",
        f"{a:.4f}",
        f"{omega_pe:.4e}",
        f"{f_pe:.4e}",
        f"{lambda_D.to(u.mm):.4f}",
        f"{cbar:.5f}",
        f"{lambdaD_bar:.5f}",
    ]
})

display(params_df)

## 2. Computational implementation

We now define a unified set of functions for:

- the cylindrical eigenvalues $X_{mn}$;
- the cold analytical branches;
- the exact warm-fluid branches;
- an approximate thermal expression;
- the dielectric functions;
- the residual equations;
- numerical extraction of the low- and high-frequency branches.

The implementation is designed so that the same notation is used throughout the notebook.

## 2B. Kinetic dispersion relation (Q_m formulation)

We now replace the fluid description with a kinetic dispersion relation
written in residual form

$$
Q_m(\bar\omega, \bar k) = 0.
$$

This formulation includes the plasma dispersion function $Z(\beta)$ and
naturally introduces complex frequencies, allowing for damping effects.

The roots of this equation will be obtained numerically.

In [ ]:
def bessel_zero(m: int, n: int) -> float:
    """Return the n-th zero of J_m."""
    return float(jn_zeros(m, n)[-1])


def omega_bar_cold(kbar, cbar, X, branch="low"):
    """
    Closed-form cold-plasma dispersion relation for cylindrical TM modes.
    """
    kbar = np.asarray(kbar, dtype=float)
    alpha = 1 + cbar**2 * (kbar**2 + X**2)
    disc = np.maximum(alpha**2 - 4 * cbar**2 * kbar**2, 0.0)

    if branch == "low":
        y = 0.5 * (alpha - np.sqrt(disc))
    else:
        y = 0.5 * (alpha + np.sqrt(disc))

    return np.sqrt(np.maximum(y, 0.0))


def omega_bar_warm_exact(kbar, cbar, X, lambdaD_bar, branch="low"):
    """
    Exact warm-fluid solution obtained by reducing the residual equation
    to a quadratic polynomial in y = omega_bar^2.
    """
    kbar = np.asarray(kbar, dtype=float)
    D = 3 * lambdaD_bar**2 * kbar**2
    alpha = 1 + D + cbar**2 * (kbar**2 + X**2)
    beta = cbar**2 * (kbar**2 * (1 + D) + X**2 * D)
    disc = np.maximum(alpha**2 - 4 * beta, 0.0)

    if branch == "low":
        y = 0.5 * (alpha - np.sqrt(disc))
    else:
        y = 0.5 * (alpha + np.sqrt(disc))

    return np.sqrt(np.maximum(y, 0.0))


def omega_bar_warm_approx(kbar, cbar, X, lambdaD_bar, branch="low"):
    """
    Alternative approximate thermal expression kept for comparison.
    """
    kbar = np.asarray(kbar, dtype=float)
    w0 = np.maximum(omega_bar_cold(kbar, cbar, X, branch=branch), 1e-12)

    term1 = (
        1
        + (3 / w0) * lambdaD_bar**2 * kbar**2
        + cbar**2 * X**2
        + cbar**2 * kbar**2
    )
    term2 = (1 + (3 / w0) * lambdaD_bar**2 * kbar**2) * cbar**2 * kbar**2
    disc = np.maximum(term1**2 - 4 * term2, 0.0)

    if branch == "low":
        y = 0.5 * (term1 - np.sqrt(disc))
    else:
        y = 0.5 * (term1 + np.sqrt(disc))

    return np.sqrt(np.maximum(y, 0.0))


def eps_cold(omega_bar):
    """Cold dielectric function."""
    return 1 - 1 / omega_bar**2


def eps_warm(omega_bar, kbar, lambdaD_bar):
    """Warm-fluid dielectric function."""
    return 1 - 1 / (omega_bar**2 - 3 * lambdaD_bar**2 * kbar**2)


def residual_cold(omega_bar, kbar, cbar, X):
    """Residual equation in the cold limit."""
    return kbar**2 - omega_bar**2 / cbar**2 + X**2 / eps_cold(omega_bar)


def residual_warm(omega_bar, kbar, cbar, X, lambdaD_bar):
    """Residual equation in the warm-fluid model."""
    return kbar**2 - omega_bar**2 / cbar**2 + X**2 / eps_warm(omega_bar, kbar, lambdaD_bar)


def _root_in_interval(func, left, right, nscan=1200):
    """
    Locate a root by scanning for a sign change and refining with Brent's method.
    """
    if right <= left:
        return np.nan

    xs = np.linspace(left, right, nscan)
    ys = np.array([func(x) for x in xs], dtype=float)

    finite = np.isfinite(ys) & (np.abs(ys) < 1e10)
    xs, ys = xs[finite], ys[finite]

    if len(xs) < 2:
        return np.nan

    for i in range(len(xs) - 1):
        y1, y2 = ys[i], ys[i + 1]

        if y1 == 0:
            return xs[i]

        if np.sign(y1) != np.sign(y2):
            try:
                return brentq(func, xs[i], xs[i + 1])
            except ValueError:
                continue

    return np.nan


def numerical_branch_cold(kbar, cbar, X, branch="low"):
    """
    Numerical extraction of a cold branch from the residual equation.
    """
    if np.isclose(kbar, 0.0) and branch == "low":
        return 0.0

    resonance = 1.0
    xmax = 1.2 * np.sqrt(1 + cbar**2 * (kbar**2 + X**2)) + 1.0
    f = lambda w: residual_cold(w, kbar, cbar, X)

    if branch == "low":
        return _root_in_interval(f, 1e-6, resonance - 1e-5)

    return _root_in_interval(f, resonance + 1e-5, xmax)


def numerical_branch_warm(kbar, cbar, X, lambdaD_bar, branch="low"):
    """
    Numerical extraction of a warm-fluid branch from the residual equation.
    """
    if np.isclose(kbar, 0.0):
        if branch == "low":
            return 0.0
        return omega_bar_warm_exact(np.array([0.0]), cbar, X, lambdaD_bar, branch="high")[0]

    resonance = np.sqrt(1 + 3 * lambdaD_bar**2 * kbar**2)
    xmax = 1.2 * np.sqrt(1 + 3 * lambdaD_bar**2 * kbar**2 + cbar**2 * (kbar**2 + X**2)) + 1.0
    f = lambda w: residual_warm(w, kbar, cbar, X, lambdaD_bar)

    if branch == "low":
        return _root_in_interval(f, 1e-6, resonance - 1e-5)

    return _root_in_interval(f, resonance + 1e-5, xmax)


def to_Hz(omega_bar_values, f_pe_hz):
    """Convert normalized frequency to Hz."""
    return omega_bar_values * f_pe_hz.to_value(u.Hz)


def kbar_to_k(kbar_values, a):
    """Convert normalized wavenumber to m^-1."""
    return np.asarray(kbar_values) / a.to_value(u.m)

In [ ]:
def _root_minimize_abs(func, left, right, nscan=800):

  xs = np.linspace(left, right, nscan)
  ys = np.array([np.abs(func(x)) for x in xs])

  idx = np.argmin(ys)
  x0 = xs[idx]

  res = minimize(lambda x: np.abs(func(x[0])), [x0])

  if res.success:
    return res.x[0]

  return np.nan



def muller_complex(func, x0, x1, x2, tol=1e-11, maxiter=1000):
  """
  Muller Numeric Method
  """
  for _ in range(maxiter):

    f0, f1, f2 = func(x0), func(x1), func(x2)

    h0 = x1 - x0
    h1 = x2 - x1
    δ0 = (f1 - f0) / h0
    δ1 = (f2 - f1) / h1

    a = (δ1 - δ0) / (h1 + h0)
    b = a * h1 + δ1
    c = f2


    if b >= 0:
      denom = b + np.sqrt(b**2 - 4 * a * c)
    else:
      denom = b - np.sqrt(b**2 - 4 * a * c)

    x3 = x2 - 2 * c / denom

    if abs(x3 - x2) < tol:
      return x3

    x0, x1, x2 = x1, x2, x3

  return x3


In [ ]:
def Z_prime_function(beta):
  return plasma_dispersion_func_deriv(beta)

def Q_m(omega_bar, kbar, cbar, X, lambdaD_bar, m):
  omega_bar = np.asarray(omega_bar)

  term1 = (cbar**2 * kbar**2) - omega_bar**2
  denom = cbar**2 * (kbar**2 + X**2) - omega_bar**2

  term1 = term1 / denom

  beta = omega_bar / (np.sqrt(2) * kbar * lambdaD_bar + 1e-12)

  Zb = Z_prime_function(beta)

  C = 1.0

  term2 = C * Zb
  denom2 = 2 * lambdaD_bar**2 * kbar**2

  term2 = term2 / denom2

  delta_m = 1
  return term1 * term2 - delta_m

def numerical_branch_Q(kbar, cbar, X, lambdaD_bar, m, branch="low"):
  f = lambda w: Q_m(w, kbar, cbar, X, lambdaD_bar, m)

  if branch == "low":
    return _root_minimize_abs(f, 1e-5, 1.5)

  return _root_minimize_abs(f, 1.5, 5.0)


def numerical_branch_muller(kbar, cbar, X, lambdaD_bar, m, guess):
  f = lambda w: Q_m(w, kbar, cbar, X, lambdaD_bar, m)

  x0 = guess * (1 - 0.05)
  x1 = guess
  x2 = guess * (1 + 0.05)

  return muller_complex(f, x0, x1, x2)



## 3. Generic branch structure and cutoff behavior

Before turning to the full cylindrical modal spectrum, it is useful to examine the branch structure in a simplified normalized setting. Replacing the cylindrical eigenvalue $X_{mn}$ by a generic transverse parameter $\rho$, one obtains

$$
\bar\omega
=
\sqrt{
\left(
\frac{1}{2}
+
\frac{\rho^2 \bar c^2}{2}
+
\frac{\bar c^2 \bar k^2}{2}
\right)
\pm
\sqrt{
\left(
\frac{1}{2}
+
\frac{\rho^2 \bar c^2}{2}
+
\frac{\bar c^2 \bar k^2}{2}
\right)^2
-
\bar c^2 \bar k^2
}
}.
$$

The corresponding cutoff is

$$
\bar\omega_{\mathrm{cutoff}} = \sqrt{1 + \rho^2 \bar c^2}.
$$

This simplified case makes the modal splitting and asymptotic behavior transparent.

In [ ]:
rho = 0.6
kbar_sym = np.linspace(-5, 5, 500)

A = 0.5 + 0.5 * cbar**2 * (rho**2 + kbar_sym**2)
B = cbar**2 * kbar_sym**2
radicand = np.maximum(A**2 - B, 0.0)

omega_plus = np.sqrt(A + np.sqrt(radicand))
omega_minus = np.sqrt(np.maximum(A - np.sqrt(radicand), 0.0))
cutoff_bar = np.sqrt(1 + cbar**2 * rho**2)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(kbar_sym, omega_plus, lw=2.5, label=r"$\bar\omega_+$")
ax.plot(kbar_sym, omega_minus, lw=2.5, label=r"$\bar\omega_-$")
ax.plot(kbar_sym, cbar * np.abs(kbar_sym), ls="--", lw=1.8, color="0.5", label=r"$\bar\omega = \bar c |\bar k|$")
ax.axhline(1.0, ls=":", lw=2.0, color="k", label=r"$\bar\omega = 1$")
ax.axhline(cutoff_bar, ls="-.", lw=1.8, color="tab:red", label=rf"$\bar\omega_{{cutoff}}={cutoff_bar:.3f}$")
ax.axvline(0, color="k", lw=0.8)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel(r"$\bar k = ka$")
ax.set_ylabel(r"$\bar\omega = \omega / \omega_{pe}$")
ax.set_title("Generic normalized branch structure")
ax.set_ylim(0, None)
ax.legend(ncol=2)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(kbar_sym, A**2, lw=2.0, label=r"$A^2$")
ax.plot(kbar_sym, B, lw=2.0, label=r"$B = \bar c^2 \bar k^2$")
ax.set_xlabel(r"$\bar k$")
ax.set_ylabel("value")
ax.set_title("Discriminant components")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
idx = np.argsort(A)
ax.plot(A[idx], A[idx], lw=2.0, label=r"$A$")
ax.plot(A[idx], np.sqrt(radicand[idx]), lw=2.0, label=r"$\sqrt{A^2-B}$")
ax.set_xlabel(r"$A$")
ax.set_ylabel("value")
ax.set_title("Comparison between $A$ and the square-root discriminant")
ax.legend()
plt.show()

## 4. Cold spectrum of cylindrical TM modes

We now evaluate the cylindrical modes directly. The indices $(m,n)$ specify the azimuthal and radial structure of each mode. The first set of spectra below shows the low- and high-frequency branches for the first few values of $m$ and $n$.

These figures establish the global modal organization before we compare analytical and numerical solutions mode by mode.

In [ ]:
kbar = np.linspace(0, 10, 350)
k_m = kbar_to_k(kbar, a)

modes_full = [(m, n) for m in [0, 1, 2] for n in [1, 2, 3]]

fig, ax = plt.subplots(figsize=(10, 6))
for m, n in modes_full:
    X = bessel_zero(m, n)
    omega_bar = omega_bar_cold(kbar, cbar, X, branch="low")
    ax.plot(k_m, to_Hz(omega_bar, f_pe), label=fr"TM$_{{{m},{n}}}$")

ax.set_xlabel(r"$k$ [m$^{-1}$]")
ax.set_ylabel("frequency [Hz]")
ax.set_title("Cylindrical TM spectrum: cold low-frequency branch")
ax.legend(ncol=3, fontsize=10)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
for m, n in modes_full:
    X = bessel_zero(m, n)
    omega_bar = omega_bar_cold(kbar, cbar, X, branch="high")
    ax.plot(k_m, to_Hz(omega_bar, f_pe), label=fr"TM$_{{{m},{n}}}$")

ax.set_xlabel(r"$k$ [m$^{-1}$]")
ax.set_ylabel("frequency [Hz]")
ax.set_title("Cylindrical TM spectrum: cold high-frequency branch")
ax.legend(ncol=3, fontsize=10)
plt.tight_layout()
plt.show()

## 5. Analytical and numerical comparison in the cold limit

A central consistency check is whether the roots obtained from the residual equation reproduce the closed-form cold dispersion relation. To make this comparison visually clear, we select three representative modes:

- TM$_{0,1}$,
- TM$_{1,1}$,
- TM$_{2,1}$.

In the following figures:

- the **solid curves** are the analytical cold branches;
- the **markers** are numerical roots obtained from the residual equation.

Agreement between both confirms that the analytical formula and the numerical implementation are solving the same spectral problem.

In [ ]:
selected_modes = [(0, 1), (1, 1), (2, 1)]
colors = {
    (0, 1): "tab:blue",
    (1, 1): "tab:orange",
    (2, 1): "tab:green",
}

kbar_num = np.linspace(0, 10, 120)
k_num = kbar_to_k(kbar_num, a)

fig, ax = plt.subplots(figsize=(10, 6))
for mode in selected_modes:
    m, n = mode
    X = bessel_zero(m, n)
    color = colors[mode]

    omega_ana = omega_bar_cold(kbar_num, cbar, X, branch="low")
    omega_num = np.array([numerical_branch_cold(kb, cbar, X, branch="low") for kb in kbar_num])

    ax.plot(k_num, to_Hz(omega_ana, f_pe), color=color, lw=2.5, label=fr"TM$_{{{m},{n}}}$ analytical")
    ax.scatter(
        k_num[::4],
        to_Hz(omega_num[::4], f_pe),
        color=color,
        s=20,
        marker="o",
        alpha=0.9,
        label=fr"TM$_{{{m},{n}}}$ numerical"
    )

ax.set_xlabel(r"$k$ [m$^{-1}$]")
ax.set_ylabel("frequency [Hz]")
ax.set_title("Cold low-frequency branch: analytical curves and numerical roots")
ax.legend(ncol=2, fontsize=10)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
for mode in selected_modes:
    m, n = mode
    X = bessel_zero(m, n)
    color = colors[mode]

    omega_ana = omega_bar_cold(kbar_num, cbar, X, branch="high")
    omega_num = np.array([numerical_branch_cold(kb, cbar, X, branch="high") for kb in kbar_num])

    ax.plot(k_num, to_Hz(omega_ana, f_pe), color=color, lw=2.5, label=fr"TM$_{{{m},{n}}}$ analytical")
    ax.scatter(
        k_num[::4],
        to_Hz(omega_num[::4], f_pe),
        color=color,
        s=20,
        marker="s",
        alpha=0.9,
        label=fr"TM$_{{{m},{n}}}$ numerical"
    )

ax.set_xlabel(r"$k$ [m$^{-1}$]")
ax.set_ylabel("frequency [Hz]")
ax.set_title("Cold high-frequency branch: analytical curves and numerical roots")
ax.legend(ncol=2, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
records = []

for m, n in selected_modes:
    X = bessel_zero(m, n)
    for branch in ["low", "high"]:
        omega_ana = omega_bar_cold(kbar_num, cbar, X, branch=branch)
        omega_num = np.array([numerical_branch_cold(kb, cbar, X, branch=branch) for kb in kbar_num])

        denom = np.maximum(np.abs(omega_ana), 1e-12)
        rel_err = np.abs(omega_ana - omega_num) / denom

        records.append({
            "Mode": f"TM({m},{n})",
            "Branch": branch,
            "Maximum relative error": np.nanmax(rel_err),
            "Mean relative error": np.nanmean(rel_err),
        })

cold_error_df = pd.DataFrame(records)
display(cold_error_df.style.format({
    "Maximum relative error": "{:.3e}",
    "Mean relative error": "{:.3e}",
}))

## 6. Thermal corrections and branch deformation

We now include finite-temperature effects through the normalized Debye length $\bar\lambda_D$.

For each selected mode, we compare four objects in the same figure:

1. the **cold branch** as a baseline;
2. the **exact warm-fluid branch** obtained analytically;
3. the **numerical roots** of the warm-fluid residual equation;
4. an **approximate thermal expression** plotted for reference.

This superposition makes it possible to identify:

- how strongly temperature shifts the dispersion curve;
- whether the main effect occurs in the low or high branch;
- how closely the approximate thermal expression reproduces the exact warm-fluid model.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for mode in selected_modes:
    m, n = mode
    X = bessel_zero(m, n)
    color = colors[mode]

    cold_low = omega_bar_cold(kbar_num, cbar, X, branch="low")
    warm_low = omega_bar_warm_exact(kbar_num, cbar, X, lambdaD_bar, branch="low")
    warm_low_approx = omega_bar_warm_approx(kbar_num, cbar, X, lambdaD_bar, branch="low")
    warm_low_num = np.array([numerical_branch_warm(kb, cbar, X, lambdaD_bar, branch="low") for kb in kbar_num])

    ax.plot(k_num, to_Hz(cold_low, f_pe), color="0.6", lw=1.7, ls="--")
    ax.plot(k_num, to_Hz(warm_low, f_pe), color=color, lw=2.5, label=fr"TM$_{{{m},{n}}}$ warm exact")
    ax.scatter(
        k_num[::4],
        to_Hz(warm_low_num[::4], f_pe),
        color=color,
        s=18,
        marker="o",
        label=fr"TM$_{{{m},{n}}}$ warm numerical"
    )
    ax.plot(k_num, to_Hz(warm_low_approx, f_pe), color=color, lw=1.2, ls=":")

ax.set_xlabel(r"$k$ [m$^{-1}$]")
ax.set_ylabel("frequency [Hz]")
ax.set_title("Thermal comparison: low-frequency branch")
ax.legend(ncol=2, fontsize=10)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))

for mode in selected_modes:
    m, n = mode
    X = bessel_zero(m, n)
    color = colors[mode]

    cold_high = omega_bar_cold(kbar_num, cbar, X, branch="high")
    warm_high = omega_bar_warm_exact(kbar_num, cbar, X, lambdaD_bar, branch="high")
    warm_high_approx = omega_bar_warm_approx(kbar_num, cbar, X, lambdaD_bar, branch="high")
    warm_high_num = np.array([numerical_branch_warm(kb, cbar, X, lambdaD_bar, branch="high") for kb in kbar_num])

    ax.plot(k_num, to_Hz(cold_high, f_pe), color="0.6", lw=1.7, ls="--")
    ax.plot(k_num, to_Hz(warm_high, f_pe), color=color, lw=2.5, label=fr"TM$_{{{m},{n}}}$ warm exact")
    ax.scatter(
        k_num[::4],
        to_Hz(warm_high_num[::4], f_pe),
        color=color,
        s=18,
        marker="s",
        label=fr"TM$_{{{m},{n}}}$ warm numerical"
    )
    ax.plot(k_num, to_Hz(warm_high_approx, f_pe), color=color, lw=1.2, ls=":")

ax.set_xlabel(r"$k$ [m$^{-1}$]")
ax.set_ylabel("frequency [Hz]")
ax.set_title("Thermal comparison: high-frequency branch")
ax.legend(ncol=2, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
records = []

for m, n in selected_modes:
    X = bessel_zero(m, n)

    for branch in ["low", "high"]:
        warm_exact = omega_bar_warm_exact(kbar_num, cbar, X, lambdaD_bar, branch=branch)
        warm_num = np.array([numerical_branch_warm(kb, cbar, X, lambdaD_bar, branch=branch) for kb in kbar_num])
        warm_approx = omega_bar_warm_approx(kbar_num, cbar, X, lambdaD_bar, branch=branch)

        err_num = np.abs(warm_exact - warm_num) / np.maximum(np.abs(warm_exact), 1e-12)
        err_approx = np.abs(warm_exact - warm_approx) / np.maximum(np.abs(warm_exact), 1e-12)

        records.append({
            "Mode": f"TM({m},{n})",
            "Branch": branch,
            "Exact vs numerical error (max)": np.nanmax(err_num),
            "Exact vs numerical error (mean)": np.nanmean(err_num),
            "Approximation deviation (max)": np.nanmax(err_approx),
            "Approximation deviation (mean)": np.nanmean(err_approx),
        })

warm_error_df = pd.DataFrame(records)
display(warm_error_df.style.format({
    "Exact vs numerical error (max)": "{:.3e}",
    "Exact vs numerical error (mean)": "{:.3e}",
    "Approximation deviation (max)": "{:.3e}",
    "Approximation deviation (mean)": "{:.3e}",
}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))


for mode in selected_modes:
  m, n = mode

  X = bessel_zero(m, n)
  color = colors[mode]

  omega_cold = omega_bar_cold(kbar_num, cbar, X, branch="low")
  omega_warm = omega_bar_warm_exact(kbar_num, cbar, X, lambdaD_bar, branch="low")

  omega_kin = np.array([
      numerical_branch_Q(kb, cbar, X, lambdaD_bar, m)
      for kb in kbar_num
  ])


  omega_kin_muller = []

  guess = 0.05

  for i, kb in enumerate(kbar_num):

    guess_k = max(guess, cbar * kb * 0.1)

    root = numerical_branch_muller(
        kb, cbar, X, lambdaD_bar, m, guess_k
    )

    if np.isfinite(root):
      guess = root

    omega_kin_muller.append(root)

  omega_kin_muller = np.array(omega_kin_muller)


  ax.plot(k_num, to_Hz(omega_cold, f_pe), ls="--", color="0.6")
  ax.plot(k_num, to_Hz(omega_warm, f_pe), color=color, lw=2, label=fr"TM$_{{{m},{n}}}$ warm")
  ax.plot(k_num, to_Hz(omega_kin_muller, f_pe), color="tab:red", lw=2, label=fr"TM$_{{{m},{n}}}$ kinetic muller")
  ax.scatter(
      k_num[::4],
      to_Hz(omega_kin[::4], f_pe),
      color = color,
      s= 20,
      marker = "o",
      label = fr"TM$_{{{m},{n}}}$ kinetic"
  )

  ax.set_xlabel(r"$k$")
  ax.set_ylabel("frequency [Hz]")
  ax.set_title("Cold vs Warm vs Kinetic dispersion")
  ax.legend(ncol=2, fontsize=10)
  plt.tight_layout()
  plt.show()

## 7. Interpretation of the results

The computations above support several conclusions.

### 7.1 Cold analytical solution and residual roots

The cold analytical branches and the numerical roots obtained from the residual equation overlap to high precision. This validates both the algebraic implementation of the closed-form solution and the numerical root-finding procedure.

### 7.2 Thermal effects

The warm-fluid corrections modify the spectrum through the factor $3\bar\lambda_D^2 \bar k^2$, which becomes increasingly relevant as the axial wavenumber grows. For the present parameter set:

- the **low-frequency branch** exhibits the clearest thermal shift;
- the **high-frequency branch** remains closer to the cold result over much of the scanned domain.

### 7.3 Approximate versus exact thermal description

The approximate thermal expression captures the general trend of the warm branch, but the direct comparison with the exact warm-fluid solution quantifies where the approximation deviates and by how much. This is useful when deciding whether an approximate expression is sufficient for parameter scans or whether the exact warm-fluid solution should be retained.

---

## 8. Extensions

The present implementation is ready to be extended in several directions:

- larger mode sets $(m,n)$;
- different densities, temperatures, and cylinder radii;
- alternative thermal closures;
- inclusion of ions in the dielectric function;
- comparison with experimental or simulated dispersion data.

## References

- J. H. Malmberg and C. B. Wharton, *Collisionless damping of electrostatic plasma waves*, Phys. Rev. Lett. **17**, 175 (1966).
- J. C. Hoyos et al., *Dispersion relation for transverse magnetic modes in a homogeneous plasma*, J. Phys.: Conf. Ser. **1247**, 012005 (2019).
- PlasmaPy documentation for `plasma_frequency` and `Debye_length`.